Build AI Agent with Microsoft Autogen

In [6]:
! pip install autogen-agentchat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [10]:
! pip install autogen-ext[openai]

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [12]:
import os 
import asyncio

import requests
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console 
from autogen_ext.models.openai import OpenAIChatCompletionClient


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

modal_client = OpenAIChatCompletionClient(
    model="gpt-4o"
)

async def get_weather(city: str) -> str:
    try:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": city,
            "appid": OPENWEATHER_API_KEY,
            "units": "metric"
        }
        response = requests.get(url, params=params)
        data = response.json()
        if response.status_code != 200 or "weather" not in data:
            return f"Could not fetch the weather for {city}"

        desc = data["weather"][0]["description"].capitalize()
        temp = data["main"]["temp"]
        location = data["name"]
        return f"{location}: {desc}: {temp}°C"
    except Exception as e:
        return f"Error fetching the weather data: {str(e)}"


agent = AssistantAgent(
    name="Weather_Agent",
    model_client=modal_client,
    tools=[get_weather],
    system_message=(
        "You are a helpful weather assistant. If the user asks about the weather, "
        "use the 'get_weather' tool to find real-time information."
    ),
    reflect_on_tool_use=True,
    model_client_stream=True
)

async def main():
    await Console(agent.run_stream(task="What is the weather in Lahore"))
    await modal_client.close()

await main()


---------- TextMessage (user) ----------
What is the weather in Lahore
---------- ToolCallRequestEvent (Weather_Agent) ----------
[FunctionCall(id='call_K11Fc55WOXXpLhO3FMh1PIhg', arguments='{"city":"Lahore"}', name='get_weather')]
---------- ToolCallExecutionEvent (Weather_Agent) ----------
[FunctionExecutionResult(content='Lahore: Clear sky: 34.99°C', name='get_weather', call_id='call_K11Fc55WOXXpLhO3FMh1PIhg', is_error=False)]
---------- ModelClientStreamingChunkEvent (Weather_Agent) ----------
The weather in Lahore is currently clear with a temperature of 34.99°C.
